# Results Analysis

In [ ]:
def default_params(): 
    return {
        'current_model' : 'M6',
        'input_path' : '/workspaces/mica/datax/causal_revision_out_results/strict',
        'output_plots' : '/workspaces/mica/notebooks/pii/correlation_plots'
    }
params = default_params()

### Imports

In [343]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [344]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

### Dataset loading

In [345]:
def load_json_dataframes(directory):
    """
    Reads all .json files from the specified directory, parses each as a pandas DataFrame,
    and returns a dictionary with file names (without extension) as keys and DataFrames as values.
    """
    dataframes = {}
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            filepath = os.path.join(directory, filename)
            key = os.path.splitext(filename)[0]
            dataframes[key] = pd.read_json(filepath)
    return dataframes

In [346]:
causal_results_dict = load_json_dataframes(f"{params['input_path']}/{params['current_model']}")

In [347]:
causal_results_dict['T1'].sort_values(by=['estimated_effect'], ascending=True)

,method_name,pearson_corr,spearman_corr,kendall_corr,estimated_effect,refutation_placebo_permute,refutation_unobserved_confounder,refutation_subset,pii_type
0,backdoor.propensity_score_matching,0.008366,0.008366,0.008366,0.003241,"{'result': {'p_value': 0.0, 'is_statistically_...","{'result': None, 'new_effect': None}","{'result': {'p_value': 0.64, 'is_statistically...",name
2,backdoor.propensity_score_matching,0.007022,0.007022,0.007022,0.006974,"{'result': {'p_value': 0.06, 'is_statistically...","{'result': None, 'new_effect': None}","{'result': {'p_value': 0.64, 'is_statistically...",username
1,backdoor.propensity_score_matching,0.068816,0.068816,0.068816,0.010989,"{'result': {'p_value': 0.98, 'is_statistically...","{'result': None, 'new_effect': None}","{'result': {'p_value': 0.76, 'is_statistically...",email


In [348]:
causal_results_dict['T1'].groupby(['pii_type']).describe()

pearson_corr                                                        \
                count      mean std       min       25%       50%       75%   
pii_type                                                                      
email             1.0  0.068816 NaN  0.068816  0.068816  0.068816  0.068816   
name              1.0  0.008366 NaN  0.008366  0.008366  0.008366  0.008366   
username          1.0  0.007022 NaN  0.007022  0.007022  0.007022  0.007022   

                   spearman_corr            ... kendall_corr            \
               max         count      mean  ...          75%       max   
pii_type                                    ...                          
email     0.068816           1.0  0.068816  ...     0.068816  0.068816   
name      0.008366           1.0  0.008366  ...     0.008366  0.008366   
username  0.007022           1.0  0.007022  ...     0.007022  0.007022   

         estimated_effect                                              \
                    count      mean std       min       25%       50%   
pii_type                                                                
email                 1.0  0.010989 NaN  0.010989  0.010989  0.010989   
name                  1.0  0.003241 NaN  0.003241  0.003241  0.003241   
username              1.0  0.006974 NaN  0.006974  0.006974  0.006974   

                              
               75%       max  
pii_type                      
email     0.010989  0.010989  
name      0.003241  0.003241  
username  0.006974  0.006974  

[3 rows x 32 columns]

### Plots

In [349]:
def plot_correlations(causal_df, treatment, correlation, method_name):
    # Transforming the DataFrame using melt
    causal_df = causal_df[causal_df['method_name'] == method_name]
    correlation_plot_df = causal_df.melt(id_vars=['pii_type'], value_vars=[correlation, 'estimated_effect'],  var_name='type', value_name='value')

    # Create the plot
    plt.figure(figsize=(10, 6))
    barplot = sns.barplot(
        palette='mako',
        data=correlation_plot_df,
        x="pii_type",
        y="value",
        hue="type",
        alpha=0.8, 
        errorbar=None)

    # Improve x-axis readability
    plt.xlabel("PII Type", fontsize=12)
    plt.ylabel("Value", fontsize=12)
    plt.title(f"{correlation} vs ATE. ASR, Model [{params['current_model']}]", fontsize=14)

    # Move the legend outside for clarity
    plt.legend(title="Metrics", bbox_to_anchor=(1.05, 1), loc='upper left')
    
    output_dir = f"{params['output_plots']}/{params['current_model']}/{method_name}"
    create_folder(output_dir)
    plt.savefig(f"{output_dir}/{treatment}_{correlation}.png", bbox_inches="tight", dpi=300)
    plt.close()



In [350]:
#method_names = ['backdoor.generalized_linear_model', 'backdoor.propensity_score_matching', 'backdoor.propensity_score_stratification', 'backdoor.propensity_score_weighting']
#method_names = ['backdoor.propensity_score_matching']

#for treatment, causal_df in causal_results_dict.items():
#    for correlation in ['pearson_corr', 'spearman_corr', 'kendall_corr']:
#        for method_name in method_names:
#            plot_correlations(causal_df, treatment, correlation, method_name)

In [351]:
causal_results_dict['T1'].columns

Index(['method_name', 'pearson_corr', 'spearman_corr', 'kendall_corr',
       'estimated_effect', 'refutation_placebo_permute',
       'refutation_unobserved_confounder', 'refutation_subset', 'pii_type'],
      dtype='str')

#### Summary

In [352]:
type_map = {'easy': 'T0', 'hard': 'T1', 'ambiguous': 'T2'}
treatment = type_map['ambiguous']

summary_df = causal_results_dict[treatment][causal_results_dict[treatment]['method_name']=='backdoor.propensity_score_matching'].copy()
summary_df = summary_df[['pii_type','pearson_corr','estimated_effect', 'refutation_placebo_permute']].sort_values(by='estimated_effect', ascending=False)
summary_df['pearson_corr'] = summary_df['pearson_corr'].round(3)
summary_df['estimated_effect'] = summary_df['estimated_effect'].round(3)
summary_df['refutation_placebo_permute'] = summary_df['refutation_placebo_permute'].map(lambda result: result['new_effect'])
summary_df['refutation_placebo_permute'] = summary_df['refutation_placebo_permute'].round(4)
summary_df.sort_values(by='pii_type', ascending=True)


,pii_type,pearson_corr,estimated_effect,refutation_placebo_permute
3,email,0.043,0.021,-0.0001
0,ip_address,0.060,0.019,-0.0007
1,key,0.011,0.019,-0.0015
2,name,0.039,0.051,0.0047
5,password,0.034,0.027,-0.0032
4,username,0.050,0.120,-0.0029
